# 🤖 Step 4: ML Alert Severity Classifier

This notebook trains an **XGBoost classifier** to predict the severity an analyst would assign:
- `critical` — Immediate action required
- `high` — Investigate within 1 hour
- `medium` — Investigate within 4 hours
- `low` — Review in daily batch
- `informational` — Candidate for auto-suppression

The model is trained on historical triage decisions, compared against a severity-only
rule baseline, registered in **Unity Catalog** via MLflow, and used to score all gold alerts.

> **Note:** training labels are synthetic and generated from a latent analyst policy
> (see `02_setup_and_generate_data`). Metrics show the model recovers that policy —
> they are not real-world accuracy claims.

In [0]:
%pip install xgboost scikit-learn "mlflow>=2.16" --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import xgboost as xgb
import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature

# Load historical triage decisions
triage_df = spark.table(TABLES["historical_triage"]).toPandas()
print(f"📊 Loaded {len(triage_df):,} historical triage decisions")

# Feature engineering
features = [
    "original_severity",
    "hour_of_day",
    "is_weekend",
    "repeat_alert_count",
    "user_risk_score",
    "asset_criticality_score",
    "historical_fp_rate",
    "geo_anomaly",
    "time_anomaly",
    "threat_intel_match",
]
bool_features = ["is_weekend", "geo_anomaly", "time_anomaly", "threat_intel_match"]

# Encode source_type as numeric
source_encoder = LabelEncoder()
triage_df["source_type_encoded"] = source_encoder.fit_transform(triage_df["source_type"])
features.append("source_type_encoded")

# Target: analyst severity classification
target = "analyst_severity"
label_encoder = LabelEncoder()
triage_df["target_encoded"] = label_encoder.fit_transform(triage_df[target])

X = triage_df[features].copy()
X[bool_features] = X[bool_features].astype(int)
y = triage_df["target_encoded"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✅ Training set: {len(X_train):,} | Test set: {len(X_test):,}")
print(f"   Classes: {list(label_encoder.classes_)}")

📊 Loaded 50,000 historical triage decisions
✅ Training set: 40,000 | Test set: 10,000
   Classes: ['critical', 'high', 'informational', 'low', 'medium']


In [0]:
def severity_rule(sev):
    if sev >= 9: return "critical"
    if sev >= 7: return "high"
    if sev >= 5: return "medium"
    if sev >= 3: return "low"
    return "informational"

baseline_pred = label_encoder.transform(X_test["original_severity"].map(severity_rule))
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_macro_f1 = f1_score(y_test, baseline_pred, average="macro")
print(f"📏 Baseline (severity-only rule): accuracy={baseline_accuracy:.4f}  macro-F1={baseline_macro_f1:.4f}")

📏 Baseline (severity-only rule): accuracy=0.4391  macro-F1=0.4419


In [0]:
# Set MLflow experiment
mlflow.set_registry_uri("databricks-uc")
experiment_name = f"/Users/{spark.sql('SELECT current_user()').first()[0]}/company_soc_alert_classifier"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="xgboost_alert_classifier_v2") as run:
    # Model parameters
    params = {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "multi:softprob",
        "num_class": len(label_encoder.classes_),
        "eval_metric": "mlogloss",
        "random_state": 42
    }

    # Log parameters
    mlflow.log_params(params)

    # Train model
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # Predictions
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("baseline_accuracy", baseline_accuracy)
    mlflow.log_metric("baseline_macro_f1", baseline_macro_f1)
    mlflow.log_metric("test_size", len(X_test))
    mlflow.log_metric("train_size", len(X_train))

    # Classification report
    report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, output_dict=True)
    for cls in label_encoder.classes_:
        mlflow.log_metric(f"f1_{cls}", report[cls]["f1-score"])
        mlflow.log_metric(f"precision_{cls}", report[cls]["precision"])
        mlflow.log_metric(f"recall_{cls}", report[cls]["recall"])

    # Log model with signature
    signature = infer_signature(X_train, y_pred)
    mlflow.xgboost.log_model(
        model,
        artifact_path="alert_classifier",
        signature=signature,
        input_example=X_train.head(5)
    )

    # Log feature importance
    importance = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    mlflow.log_table(importance, artifact_file="feature_importance.json")

    print(f"✅ Model trained successfully!")
    print(f"   Run ID: {run.info.run_id}")
    print(f"\n📊 Model vs Baseline (held-out test set):")
    print(f"   {'':22s} {'accuracy':>9s} {'macro-F1':>9s}")
    print(f"   {'Severity-only rule':22s} {baseline_accuracy:9.4f} {baseline_macro_f1:9.4f}")
    print(f"   {'XGBoost classifier':22s} {accuracy:9.4f} {macro_f1:9.4f}")
    print(f"\n📊 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    print(f"📊 Confusion Matrix (rows = analyst label, cols = predicted):")
    display(pd.DataFrame(confusion_matrix(y_test, y_pred),
                         index=label_encoder.classes_, columns=label_encoder.classes_).reset_index())

    print(f"\n📊 Feature Importance:")
    display(importance)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
/local_disk0/.ephemeral_nfs/envs/pythonEnv-6e3404ca-46f8-49e2-98ee-9a115220a9dc/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs

✅ Model trained successfully!
   Run ID: 37413bdd501d4d22b6329380f02b251a

📊 Model vs Baseline (held-out test set):
                           accuracy  macro-F1
   Severity-only rule        0.4391    0.4419
   XGBoost classifier        0.6720    0.6780

📊 Classification Report:
               precision    recall  f1-score   support

     critical       0.81      0.74      0.77      1556
         high       0.59      0.63      0.61      1951
informational       0.80      0.81      0.80      2121
          low       0.59      0.60      0.60      2089
       medium       0.61      0.60      0.60      2283

     accuracy                           0.67     10000
    macro avg       0.68      0.68      0.68     10000
 weighted avg       0.67      0.67      0.67     10000

📊 Confusion Matrix (rows = analyst label, cols = predicted):


index,critical,high,informational,low,medium
critical,1153,389,0,0,14
high,266,1227,0,15,443
informational,0,0,1712,396,13
low,0,18,410,1262,399
medium,11,433,13,460,1366



📊 Feature Importance:


feature,importance
original_severity,0.33805126
threat_intel_match,0.20050469
asset_criticality_score,0.1262247
geo_anomaly,0.09368586
time_anomaly,0.06844976
user_risk_score,0.04736942
historical_fp_rate,0.038938466
hour_of_day,0.036871325
repeat_alert_count,0.021730967
source_type_encoded,0.014103612


In [0]:
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.alert_severity_classifier"

# Register the model
model_uri = f"runs:/{run.info.run_id}/alert_classifier"
registered_model = mlflow.register_model(model_uri, model_name)

print(f"✅ Model registered in Unity Catalog: {model_name}")
print(f"   Version: {registered_model.version}")

# Set alias for production
from mlflow import MlflowClient
client = MlflowClient()
client.set_registered_model_alias(model_name, "production", registered_model.version)
print(f"   Alias 'production' set to version {registered_model.version}")

Registered model 'rajarora_febar.soc_alert_triage.alert_severity_classifier' already exists. Creating a new version of this model...
2026/10/12 21:50:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 37413bdd501d4d22b6329380f02b251a has no artifacts at artifact path 'alert_classifier', registering model based on models:/m-0f2f6cddd3d44c0bbfa6b42f9310047e instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '7' of model 'rajarora_febar.soc_alert_triage.alert_severity_classifier': https://<workspace-host>/explore/data/models/rajarora_febar/soc_alert_triage/alert_severity_classifier/version/7?o=<workspace-id>


✅ Model registered in Unity Catalog: rajarora_febar.soc_alert_triage.alert_severity_classifier
   Version: 7
   Alias 'production' set to version 7


In [0]:
# Every feature is derived from real pipeline data — no simulated values.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold = spark.table(TABLES["enriched_alerts"])

# historical_fp_rate: share of past triage decisions that were FP/benign, per customer + source
fp_rates = (
    spark.table(TABLES["historical_triage"])
    .groupBy("customer_id", "source_type")
    .agg(F.avg(F.when(F.col("triage_outcome").isin("false_positive", "benign"), 1).otherwise(0))
         .alias("historical_fp_rate"))
)

entity = F.coalesce(F.col("username"), F.col("asset_id"))
scoring_sdf = (
    gold
    .join(fp_rates, ["customer_id", "source_type"], "left")
    .withColumn("ts", F.to_timestamp("event_timestamp"))
    .select(
        "event_id",
        "source_type",
        F.col("severity_score").alias("original_severity"),
        F.hour("ts").alias("hour_of_day"),
        F.dayofweek("ts").isin(1, 7).cast("int").alias("is_weekend"),
        # repeat alerts: same asset + source on the same day
        (F.count("*").over(Window.partitionBy("asset_id", "source_type", "event_date")) - 1)
            .alias("repeat_alert_count"),
        # entity risk: % of this user's (or asset's) events that are high severity
        F.round(F.avg((F.col("severity_score") >= 7).cast("int")).over(Window.partitionBy(entity)) * 100, 1)
            .alias("user_risk_score"),
        F.col("criticality_score").alias("asset_criticality_score"),
        F.coalesce("historical_fp_rate", F.lit(0.6)).alias("historical_fp_rate"),
        F.coalesce(F.col("geo_location").isin("CN", "RU", "NG"), F.lit(False)).cast("int").alias("geo_anomaly"),
        (F.hour("ts") < 6).cast("int").alias("time_anomaly"),
        F.col("threat_intel_match").cast("int").alias("threat_intel_match"),
    )
)

scoring_pdf = scoring_sdf.toPandas()
scoring_pdf["source_type_encoded"] = source_encoder.transform(scoring_pdf["source_type"])
print(f"✅ Built features for {len(scoring_pdf):,} gold alerts")
display(scoring_pdf[features].describe().T.reset_index())

✅ Built features for 500,000 gold alerts


index,count,mean,std,min,25%,50%,75%,max
original_severity,500000.0,5.50638,2.7186780019009524,1.0,3.0,6.0,8.0,10.0
hour_of_day,500000.0,11.49658,6.924184009126879,0.0,5.0,12.0,17.0,23.0
is_weekend,500000.0,0.265776,0.4417459768951113,0.0,0.0,0.0,1.0,1.0
repeat_alert_count,500000.0,0.836924,0.9803999901021426,0.0,0.0,1.0,1.0,8.0
user_risk_score,500000.0,40.9181572,11.141790475281706,0.0,33.3,42.5,48.3,100.0
asset_criticality_score,500000.0,2.5058,1.119673553915486,1.0,2.0,3.0,4.0,4.0
historical_fp_rate,500000.0,0.6989919674568307,0.04791488583305113,0.5652173913043478,0.6633663366336634,0.7011494252873564,0.7326732673267327,0.8481012658227848
geo_anomaly,500000.0,0.07505,0.2634722686266482,0.0,0.0,0.0,0.0,1.0
time_anomaly,500000.0,0.250458,0.43327723883713454,0.0,0.0,0.0,1.0,1.0
threat_intel_match,500000.0,0.027442,0.16336765289977095,0.0,0.0,0.0,0.0,1.0


In [0]:
# Load the production model
loaded_model = mlflow.xgboost.load_model(f"models:/{model_name}@production")

# Predict
prediction_probs = loaded_model.predict_proba(scoring_pdf[features])
scoring_pdf["ml_predicted_severity"] = label_encoder.inverse_transform(np.argmax(prediction_probs, axis=1))
scoring_pdf["ml_confidence"] = np.round(np.max(prediction_probs, axis=1), 4)

# Save predictions table
predictions_spark = spark.createDataFrame(scoring_pdf[["event_id", "ml_predicted_severity", "ml_confidence"]])
predictions_spark.write.mode("overwrite").saveAsTable(TABLES["alert_predictions"])

print(f"✅ Scored {len(scoring_pdf):,} alerts with ML model")
print(f"\n📊 Prediction Distribution:")
spark.sql(f"""
    SELECT ml_predicted_severity,
           COUNT(*) as count,
           ROUND(AVG(ml_confidence), 4) as avg_confidence
    FROM {TABLES['alert_predictions']}
    GROUP BY ml_predicted_severity
    ORDER BY count DESC
""").display()

✅ Scored 500,000 alerts with ML model

📊 Prediction Distribution:


ml_predicted_severity,count,avg_confidence
informational,119518,0.8154
medium,118059,0.5684
low,117365,0.5775
high,95260,0.5862
critical,49798,0.7907


In [0]:
# How much of the queue could analysts deprioritize or auto-suppress?
spark.sql(f"""
    SELECT
        COUNT(*) AS total_alerts,
        SUM(CASE WHEN p.ml_predicted_severity IN ('critical', 'high') THEN 1 ELSE 0 END) AS needs_analyst_now,
        SUM(CASE WHEN p.ml_predicted_severity = 'informational' AND p.ml_confidence >= 0.8 THEN 1 ELSE 0 END) AS auto_suppress_candidates,
        ROUND(SUM(CASE WHEN p.ml_predicted_severity IN ('critical', 'high') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_needs_analyst_now,
        ROUND(SUM(CASE WHEN e.threat_intel_match AND p.ml_predicted_severity IN ('critical', 'high') THEN 1 ELSE 0 END) * 100.0
              / NULLIF(SUM(CASE WHEN e.threat_intel_match THEN 1 ELSE 0 END), 0), 1) AS pct_threat_intel_hits_prioritized
    FROM {TABLES['enriched_alerts']} e
    JOIN {TABLES['alert_predictions']} p USING (event_id)
""").display()

total_alerts,needs_analyst_now,auto_suppress_candidates,pct_needs_analyst_now,pct_threat_intel_hits_prioritized
500000,145058,71614,29.0,100.0


## ✅ ML Model Complete!

### What was built:
- **XGBoost classifier** trained on 50K historical triage decisions
- **Severity-only rule baseline** for comparison
- **MLflow experiment** with full tracking (params, metrics, confusion matrix, feature importance)
- **Model registered** in Unity Catalog with `production` alias
- **500K alerts scored** using features derived from the gold layer and triage history

**Next step**: Run `05_genai_summaries` to generate AI investigation summaries